This is a text Markdown Cell. This is where we write stuff that involve decriptions, definitions, and explanations.

Remember:
Each Markdown Cell can only contain, at Max, 100 words. And in total, all of the markdown cells that we have created, combined, cannot exceed 5000 words.

In [ ]:
This is a code cell. Notice how words such as and, or, if, else, and else if, glow?
This is because it matches Python's syntax. 

Remember:
For each code cell, we can only have: 30 Lines Of Code Max (Excluding Blank Spaces) per Cell.
And we cannot have any comments either as the markdown cells will be used for explaining the code instead.

## Step 1: Import Libraries

In [13]:
import pandas as pd 
import numpy as np 
import csv
from sklearn.model_selection import train_test_split
SEED = 55

Import libraries for loading and cleaning data as well as splitting the dataset

In [14]:
raw = pd.read_csv("dataset_Cheat_Detection.csv", sep=";", decimal=",")

Read the file and define that ; is a separator and , is the decimal point because the dataset has European format

In [36]:
with open('cleanedDataset.csv', 'w', newline = '', encoding='utf-8') as file:
    cleaned = raw.drop(columns = ['timestamp', 'isAimbot_max', 'isVidaInfinita_max', 'isMunicionInfinita_max', 'isSuperVelocidad_max'])
    cleaned.to_csv('cleanedDataset.csv', index=False)


Removed the **Timestamp** Column because nearly every row has a unique value, so it works more as an identifier rather than a feature. The model may end up learning a time-based pattern instead of looking at the other feature values to decide whether cheating occurred or not.

**isAimbot_max, isVidaInfinita_max, isMunicionInfinita_max, and isSuperVelocidad_max** were also removed because they are sub-labels of **es_trampa_max**, which is 1 (Cheating) whenever at least one of them is true. Keeping them would leak the target to the model, and they would not be available in a real-world scenario.

In [22]:
cleaned.isna().sum()

angle_error_mean          0
angle_error_max           0
speed_mean                0
speed_max                 0
is_shot_sum               0
is_hit_sum                0
is_received_damage_sum    0
health_mean               0
ammo_left_min             0
ammo_max                  0
es_trampa_max             0
accuracy_rate             0
infinite_ammo_ratio       0
dtype: int64

Check if there are any cells with missing values that may affect the model's training or lead to errors during preprocessing.

In [37]:
dup_count = cleaned.duplicated().sum()
print ("Duplicates: ", dup_count)
if dup_count > 0:   
    cleaned.drop_duplicates(subset=None, inplace=True)

print(f'Before: {len(raw)} rows')
print(f'Target: {len(cleaned)} rows')

Duplicates:  60
Before: 3584 rows
Target: 3524 rows


Check if there are any duplicated rows that could lead to the model developing a bias and also preventing a leakage. There were 60 duplicates found since we removed the Timestamp column.

In [85]:
rawAnalysis = raw.describe()
cleanedAnalysis = cleaned.describe()

compare = pd.concat([rawAnalysis, cleanedAnalysis], axis = 1, keys=['Raw Dataset', 'Cleaned Dataset'])
compare = compare.swaplevel(0, 1, axis = 1).sort_index(axis = 1)

label = 'es_trampa_max'
raw_counts = raw[label].value_counts()
cleaned_counts = cleaned[label].value_counts()

raw_pct = raw[label].value_counts(normalize=True) * 100
cleaned_pct = cleaned[label].value_counts(normalize=True) * 100

label_compare = pd.concat([raw_counts, raw_pct, cleaned_counts, cleaned_pct], axis=1, keys=['Raw Count', 'Raw %', 'Cleaned Count', 'Cleaned %'])

rawClassMeans = raw[cleaned.columns].groupby("es_trampa_max").mean(numeric_only=True).T
cleanedClassMeans = cleaned.groupby("es_trampa_max").mean().T

classCompare = pd.concat([rawClassMeans, cleanedClassMeans], axis=1,
                         keys=["Raw Dataset", "Cleaned Dataset"])
classCompare = classCompare.swaplevel(1, 1, axis=1).sort_index(axis=1)

print(compare, "\n\n")
print(label_compare, "\n\n")
print(classCompare)

        accuracy_rate                ammo_left_min               \
      Cleaned Dataset  Raw Dataset Cleaned Dataset  Raw Dataset   
count     3524.000000  3584.000000     3524.000000  3584.000000   
mean         0.460943     0.453227        0.360510     0.355521   
std          0.380616     0.382022        0.407238     0.406748   
min          0.000000     0.000000        0.000000     0.000000   
25%          0.000000     0.000000        0.000000     0.000000   
50%          0.454545     0.440972        0.125000     0.125000   
75%          0.818182     0.800000        0.750000     0.750000   
max          1.000000     1.000000        1.000000     1.000000   

             ammo_max              angle_error_max               \
      Cleaned Dataset  Raw Dataset Cleaned Dataset  Raw Dataset   
count     3524.000000  3584.000000     3524.000000  3584.000000   
mean         0.596481     0.587542        0.130310     0.128129   
std          0.403681     0.407235        0.253426     0.2518

#### Comparison of Raw and Cleaned Datasets

A summary analysis using describe() was implemented for both the Raw and Cleaned datasets to compare changes. The cleaned dataset has fewer rows, from 3584 to 3524 because 60 duplicates were removed after dropping the timestamp and the four other sub-labels. 

The table confirms that the statistics are nearly unchanged. Every feature mean shifted less than 0.01 witha standard deviation of less than 0.004. The medians stayed relatively the same. This shows that removing the duplicate rows and irrelevant columns did not distort the distribution of the features and the balance between Cheating and Not Cheating

The label **es_trampa_max** values were also counted and showed that the ratio of Cheating and Not Cheating classes were preserved. Cheating went from 48.35% in the Raw Dataset to 48.52% in the Cleaned Dataset. Accuracy would also be a reliable evaluation metric.

#### Observations and Challenges
1. Balanced classes: The split is very close to 50/50, so the model would not require adding or subtracting samples or instances.
2. Skewed Features: Observing the dataset, many features have a lot of zeros, resulting in a positive skew distribution. For example, **angle_error_mean** has a mean of 0.093, but its median is at 0.007. Most normal players stay near zero, and the higher values mostly come from cheaters.
3. Gap between Classes: The mean for Cheaters is much higher on multiple features compared to Non-Cheaters. The largest differences are:\
       a. **ammo_left_min** (0.517 vs 0.213)\
       b. **infinite_ammo_ratio** (0.275 vs 0.038)\
       c. **angle_error_max** (0.252 vs 0.015)\
    These three feature may be the biggest indicators for Cheating, which suggests the model can perform well
       
   
